# UXsim 学習ノートブック 1 — 基本の使い方とオンランプ合流の再現

このノートブックは、[UXsim](https://github.com/toruseo/UXsim) を使って交通流シミュレーションの
基礎を学ぶための教材です。以下の4パートで構成されています。

1. **Part A**: UXsim公式READMEのY字ネットワーク例を動かし、基本的なAPIの使い方を確認する
2. **Part B**: 高速道路の単純な1本道リンク＋オンランプ合流1箇所のネットワークを自作し、
   合流部で渋滞が発生する様子を再現する
3. **Part C**: シミュレーション結果から時空間図（x-t図）と速度プロファイルを可視化する
4. **Part D**: 各ステップで使われている理論（Newellの簡略化追従モデル、Incremental Node Model、
   基本図の考え方）を、コードと対応させながら振り返る

理論の詳細な数式・直感的な説明は `docs/theory.md` にまとめてあります。
このノートブックでは「その理論がコードのどこで・どう使われているか」を実際に手を動かして確認します。

> 前提: `pip install uxsim` 済みの仮想環境（`venv/`）でこのノートブックを実行してください。
> UXsimはPython 3.10以上が必要です。


In [ ]:
# 動作確認: UXsimと主要ライブラリのインポート
import uxsim
from uxsim import World
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("uxsim version:", uxsim.__version__)


---
## Part A: Y字ネットワーク例（UXsim公式README準拠）

まずは公式READMEのサンプルをそのまま動かし、UXsimの基本的な使い方（ノード・リンクの定義、
OD交通需要の設定、シミュレーション実行、結果の確認）を確認します。

### ネットワーク構成
- `orig1`, `orig2`: 2つの起点ノード
- `merge`: 合流ノード（Y字の合流点）
- `dest`: 終点ノード
- `link1` (orig1→merge), `link2` (orig2→merge), `link3` (merge→dest)

### ここで使われている理論（先取り）
- 各リンクは `free_flow_speed`（自由流速度）と `jam_density`（ジャム密度）を持ち、
  これらが三角形の基本図（Fundamental Diagram）を決定します（`docs/theory.md` 1章）。
- リンク内の車の動きは Newell の簡略化追従モデルに従います（`docs/theory.md` 2章）。
- `merge` ノードでは、2本のリンクから来た車を Incremental Node Model が捌きます（`docs/theory.md` 3章）。


In [ ]:
# シミュレーション本体（World）の定義
# 単位は秒(s)とメートル(m)に統一されている
W = World(
    name="",       # シナリオ名
    deltan=5,      # プラトゥーン（車両集団）サイズ。5台をまとめて1単位として計算することで高速化する
    tmax=1200,     # シミュレーション総時間 [s]
    print_mode=1, save_mode=1, show_mode=1,   # 各種オプション（ログ出力・保存・表示）
    random_seed=0  # 乱数シード（合流時の車両選択などに使われる）
)

# ノードの作成: 名前, x座標, y座標（座標は可視化専用で、シミュレーション結果には影響しない）
W.addNode(name="orig1", x=0, y=0)
W.addNode(name="orig2", x=0, y=2)
W.addNode(name="merge", x=1, y=1)
W.addNode(name="dest", x=2, y=1)

# リンクの作成
# free_flow_speed: 自由流速度 [m/s]、number_of_lanes: 車線数
# これらの値から、内部的に jam_density=0.2 [veh/m] とあわせて三角形基本図が自動計算される
W.addLink(name="link1", start_node="orig1", end_node="merge", length=1000, free_flow_speed=20, number_of_lanes=1)
W.addLink(name="link2", start_node="orig2", end_node="merge", length=1000, free_flow_speed=20, number_of_lanes=1)
W.addLink(name="link3", start_node="merge", end_node="dest", length=1000, free_flow_speed=20, number_of_lanes=1)

# OD交通需要の設定: 起点, 終点, 発生開始時刻, 発生終了時刻, 発生率(veh/s)
W.adddemand(orig="orig1", dest="dest", t_start=0, t_end=1000, flow=0.45)
W.adddemand(orig="orig2", dest="dest", t_start=400, t_end=1000, flow=0.6)


In [ ]:
# シミュレーション実行
# 内部では 1タイムステップ(=deltan*reaction_time 秒)ごとに
#   1) 各Vehicleがcarfollow()でNewellモデルに従って進む
#   2) 各Nodeがtransfer()でIncremental Node Modelに従って車を次リンクへ受け渡す
# という2段階の更新が繰り返される
W.exec_simulation()


In [ ]:
# 結果サマリの表示
W.analyzer.print_simple_stats()


`link2`（orig2発、400秒〜合流開始）の需要が加わった400秒以降、`merge`ノードでの
奪い合いが激しくなり、`link1`・`link2`側で待ち行列（渋滞）が発生しやすくなります。
下のネットワークスナップショットで、時刻ごとの混雑状況（線の太さ=交通量、色=速度）を確認しましょう。


In [ ]:
# ネットワークスナップショットの可視化
# 線が太いほど交通量が多く、色が暗い（濃い）ほど速度が低い＝渋滞していることを示す
for t in [100, 600, 800]:
    W.analyzer.network(t=t, detailed=1, network_font_size=12)


In [ ]:
# 時空間図（x-t図）: 横軸が時刻、縦軸がorig1→merge→destの経路に沿った距離
# 各線が1台（1プラトゥーン）の車両軌跡。線の傾き = 走行速度。
# 傾きが緩やかになっている区間 = 速度が落ちている = 渋滞していることを意味する
W.analyzer.time_space_diagram_traj_links([["link1", "link3"]])


**読み方のポイント**: 傾きが急（ほぼ一定）な部分は自由流（Newellモデルの
`x(t+τ) = x(t) + u・τ` が支配的）、傾きが緩やかになる・線が詰まって見える部分は
車間制約（`x_leader(t) - δ` 側）が効いている＝渋滞している状態です。


---
## Part B: 高速道路のオンランプ合流モデル — 渋滞発生の再現

ここからは自作のネットワークです。高速道路の本線1本に、オンランプ（合流車線）が
1箇所つながる、という最もシンプルな合流ボトルネックのシナリオを作ります。

### ネットワーク構成

```
mainline_up (本線上流) --\
                          >-- merge_node --- mainline_down (本線下流) ---> dest
ramp (オンランプ) --------/
```

- `mainline_up`: 本線上流リンク（十分な容量があり、単独では渋滞しない）
- `ramp`: オンランプリンク（本線に合流）
- `merge_node`: 合流ノード（Incremental Node Modelが働く）
- `mainline_down`: 本線下流リンク。**本線上流とオンランプの合計需要が、
  下流リンクの容量を上回るように意図的に設定**し、合流部で渋滞が発生する状況を作る

### ポイント: なぜ「単純な1本道」でも渋滞が起きるのか
下流リンク自体の自由流容量は上流と同じでも、**合流によって流入需要が容量を超えると**、
Incremental Node Modelが受け入れを制限し、本線・オンランプ双方に待ち行列ができます。
これは実際の高速道路のオンランプ渋滞（サグ部や合流部のボトルネック）の基本メカニズムです。


In [ ]:
# オンランプ合流シナリオの定義
W2 = World(
    name="onramp_merge",
    deltan=5,
    tmax=3000,
    print_mode=1, save_mode=1, show_mode=1,
    random_seed=42
)

# ノード配置（x, yは可視化用の座標。オンランプが斜めから合流するイメージ）
W2.addNode("mainline_orig", 0, 0)     # 本線の起点
W2.addNode("merge_node",    5, 0)     # 合流ノード
W2.addNode("dest",          9, 0)     # 終点
W2.addNode("ramp_orig",     3, -2)    # オンランプの起点

# 本線上流リンク: 十分に長く、自由流で流れる区間
W2.addLink("mainline_up", "mainline_orig", "merge_node",
           length=3000, free_flow_speed=25, number_of_lanes=2, jam_density=0.12)

# オンランプリンク: 短めで、本線に合流する
W2.addLink("ramp", "ramp_orig", "merge_node",
           length=500, free_flow_speed=15, number_of_lanes=1, jam_density=0.12,
           merge_priority=0.5)   # 本線側は明示していないのでデフォルト値(=1)より低優先=本線優先の合流

# 本線下流リンク: 車線数を1車線分減らし、意図的にボトルネックにする
# （本線2車線＋ランプ1車線がここでは1.x車線分の容量しかない、という状況を再現）
W2.addLink("mainline_down", "merge_node", "dest",
           length=4000, free_flow_speed=25, number_of_lanes=1, jam_density=0.12)

# 本線上流リンクにも merge_priority を明示しておく（オンランプより高優先=本線優先）
W2.get_link("mainline_up").merge_priority = 1.0


**`merge_priority` について（Incremental Node Modelの直感）**:
下流の受け入れ容量が不足したとき、`merge_node` は流入候補の車を
`merge_priority` の比率に応じて確率的に選びます。本線を1.0、オンランプを0.5に
設定しているため、混雑時はおよそ2:1の比率で本線側が優先的に合流ノードを通過できます
（実際の高速道路で本線優先が原則であることに対応させています）。


In [ ]:
# OD交通需要の設定
# 本線需要: 常に一定量を流す
W2.adddemand(orig="mainline_orig", dest="dest", t_start=0, t_end=2400, flow=0.55)

# オンランプ需要: 600〜1800秒の間だけ、ラッシュ的に増加させる
# 本線0.55 + ランプ0.35 = 0.90 veh/s が、mainline_down(1車線)の容量を上回るように調整している
W2.adddemand(orig="ramp_orig", dest="dest", t_start=600, t_end=1800, flow=0.35)


In [ ]:
# mainline_down の容量を事前に確認しておく
# capacity = u*w*kappa / (u+w) （三角形基本図の頂点＝最大流率）
link_down = W2.get_link("mainline_down")
print(f"mainline_down capacity      : {link_down.capacity:.3f} veh/s")
print(f"mainline_down free-flow spd : {link_down.u} m/s")
print(f"mainline_down jam density   : {link_down.kappa} veh/m")
print(f"mainline_down critical dens : {link_down.k_star:.4f} veh/m")
print()
print("本線+ランプの合計需要(ピーク時) = 0.55 + 0.35 =", 0.55 + 0.35, "veh/s")
print("→ mainline_downの容量を上回っていれば、合流部での渋滞発生が期待される")


In [ ]:
# シミュレーション実行
W2.exec_simulation()
W2.analyzer.print_simple_stats()


`average_delay`（平均遅れ）や `delay_ratio` が0より大きく出ていれば、
自由流時間より実際の旅行時間が長くなった＝渋滞が発生したことを意味します。
次のセルで、実際にどこで・いつ渋滞が起きたかを時空間図で確認します。


In [ ]:
# 本線に沿った時空間図（mainline_up → mainline_down）
# 合流ノード付近（mainline_upの下流端）から渋滞が始まり、
# 時間とともに上流（mainline_up側）へ伸びていく様子（渋滞の後方伝播）が見えるはず
W2.analyzer.time_space_diagram_traj_links([["mainline_up", "mainline_down"]], figsize=(12, 5))


In [ ]:
# オンランプ側の時空間図も確認
W2.analyzer.time_space_diagram_traj_links([["ramp"]], figsize=(12, 4))


**理論との対応**: `mainline_up` の傾きが時間の経過とともに寝てくる
（＝速度が落ちる）のは、下流の `merge_node` で捌ききれない車が
`x_cong = x_leader - δ` の制約（Newellモデルの車間制約項）に引っかかり始めるためです。
この「詰まり」が1台ずつ後ろに伝わっていくことで、`docs/theory.md` で説明した
渋滞波速度 `w` での後方伝播が再現されています。


---
## Part C: 時空間図・速度プロファイルの可視化

`W.analyzer.time_space_diagram_traj_links()` は車両軌跡ベースの時空間図でしたが、
ここでは `link_traffic_state_to_pandas()` を使い、**空間を格子状に区切った
密度・流率・速度のヒートマップ**（Edieの一般化定義に基づく集計値）を自作します。
これはETC2.0プローブデータなど「区間ごとの平均速度の時系列」と比較しやすい形式です。


In [ ]:
# リンクごとの交通状態（q: 流率, k: 密度, v: 速度）を格子状に取得
df_state = W2.analyzer.link_traffic_state_to_pandas()
df_state.head()


In [ ]:
def plot_speed_heatmap(df_state, link_name, ax=None):
    """指定したリンクの速度(v)を時空間ヒートマップ(x-t図)として描画する。
    横軸: 時刻 t [s] / 縦軸: リンク起点からの距離 x [m] / 色: 速度 v [m/s]
    """
    df_link = df_state[df_state["link"] == link_name]
    pivot = df_link.pivot(index="x", columns="t", values="v")

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.pcolormesh(pivot.columns, pivot.index, pivot.values,
                        shading="auto", cmap="RdYlGn", vmin=0)
    ax.set_xlabel("時刻 t [s]")
    ax.set_ylabel(f"距離 x [m]  (link={link_name})")
    ax.set_title(f"速度の時空間分布 (x-t図): {link_name}")
    plt.colorbar(im, ax=ax, label="速度 v [m/s]")
    return ax

fig, axes = plt.subplots(2, 1, figsize=(11, 8))
plot_speed_heatmap(df_state, "mainline_up", ax=axes[0])
plot_speed_heatmap(df_state, "mainline_down", ax=axes[1])
plt.tight_layout()
plt.show()


緑（高速）から赤（低速）へのグラデーションで、渋滞（赤い領域）が
`merge_node` 付近（`mainline_up`の下流端、`mainline_down`の上流端）から発生し、
時間とともに `mainline_up` の上流方向へ広がっていく様子が見えるはずです。
これがまさに Newell モデルによる渋滞の後方伝播です。


In [ ]:
def plot_speed_profile_at_time(df_state, link_name, times, ax=None):
    """指定した複数の時刻について、リンク内の位置xごとの速度プロファイルを重ねて描画する。
    基本図でいう「ある瞬間の空間分布」を切り出したもの。
    """
    df_link = df_state[df_state["link"] == link_name]
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4))
    for t in times:
        # 最も近い時刻ステップを取得
        nearest_t = df_link["t"].iloc[(df_link["t"] - t).abs().argsort().iloc[0]]
        df_t = df_link[df_link["t"] == nearest_t].sort_values("x")
        ax.plot(df_t["x"], df_t["v"], marker="o", markersize=3, label=f"t={nearest_t:.0f}s")
    ax.set_xlabel("距離 x [m]")
    ax.set_ylabel("速度 v [m/s]")
    ax.set_title(f"速度プロファイル: {link_name}")
    ax.legend()
    return ax

fig, ax = plt.subplots(figsize=(9, 4))
plot_speed_profile_at_time(df_state, "mainline_up", times=[300, 900, 1500, 2100], ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
def plot_speed_timeseries_at_position(df_state, link_name, positions, ax=None):
    """仮想の「定点観測（ループ検知器）」のように、指定した位置xでの速度の時系列を描画する。
    実データ（ETC2.0プローブなど）の特定区間の速度時系列と直接比較しやすい形式。
    """
    df_link = df_state[df_state["link"] == link_name]
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 4))
    for x in positions:
        nearest_x = df_link["x"].iloc[(df_link["x"] - x).abs().argsort().iloc[0]]
        df_x = df_link[df_link["x"] == nearest_x].sort_values("t")
        ax.plot(df_x["t"], df_x["v"], label=f"x={nearest_x:.0f}m")
    ax.set_xlabel("時刻 t [s]")
    ax.set_ylabel("速度 v [m/s]")
    ax.set_title(f"定点速度の時系列: {link_name}")
    ax.legend()
    return ax

fig, ax = plt.subplots(figsize=(10, 4))
plot_speed_timeseries_at_position(df_state, "mainline_up", positions=[0, 1000, 2000, 2900], ax=ax)
plt.tight_layout()
plt.show()


`x=2900`（合流ノード直前）の速度が最初に落ち込み、`x=0`（上流端）に近い
地点ほど速度低下が遅れて現れることが分かります。この「遅れ」の大きさが、
渋滞波速度 `w` に対応する伝播時間です。実データと比較する際は、この
`plot_speed_timeseries_at_position()` の出力形式（時刻・速度の時系列）を
ETC2.0プローブデータの形式に合わせるのが最も素直な比較方法になります
（`scripts/data_loader.py` 参照）。


---
## Part D: まとめ — 使われている理論の振り返り

| やったこと | 使われていた理論 | 対応するUXsimのコード |
|---|---|---|
| `addLink(free_flow_speed=..., jam_density=..., number_of_lanes=...)` | 三角形基本図（Fundamental Diagram） | `Link.__init__` の `u, kappa, w, capacity` 計算 |
| シミュレーション中の車両の移動 | Newellの簡略化追従モデル | `Vehicle.carfollow()` |
| `merge_node` での本線・オンランプの取り合い | Incremental Node Model | `Node.transfer()`、`merge_priority` |
| 時空間図での渋滞後方伝播の確認 | 渋滞波速度 `w = 1/(τ・kappa)` | `link_traffic_state_to_pandas()` の `v` 列 |

理論の数式・直感的な説明の詳細は `docs/theory.md` を参照してください。

### 発展課題（次にやること）
- `merge_priority` の値を変えて、本線とオンランプの取り合いのバランスがどう変わるか実験する
- `mainline_down` の `number_of_lanes` を2に戻し、渋滞が解消することを確認する（ボトルネックの正体を切り分ける）
- 複数経路のあるネットワークを作り、動的利用者均衡（DUO）による経路選択の収束を観察する
- 自分の保有するETC2.0プローブデータを `data/raw/` に配置し、`scripts/data_loader.py` を
  拡張して、このノートブックの `plot_speed_timeseries_at_position()` の出力と重ね描きして比較する
